# Local Mode Walkthrough (No Docker)

This notebook runs the evaluation of this module on your Mac without Docker or Harbor. It uses the same tasks, the same agent (Codex CLI on Amazon Bedrock), the same hidden tests and the same judges as notebooks 02 and 03. Only the way each trial is isolated is different.

You run the commands of [TESTER-GUIDE.md](TESTER-GUIDE.md) from this notebook, one step at a time, and look at what each step produced. The default path takes about 12 minutes and costs under $0.50 in Amazon Bedrock. The full set of 12 trials (about 38 minutes, about $3) is off until you turn it on in section 8.

> **Start Jupyter from a plain terminal.** The agent runs inside a macOS sandbox (`sandbox-exec`, through `srt`). macOS does not let a process that is already inside a sandbox start another one. So start Jupyter from Terminal.app or iTerm, not from a terminal inside an agent tool, an IDE sandbox or another sandbox. The check in section 2 finds this and tells you what to do.
>
> From the root of the repository:
>
> ```bash
> cd "Workload Specific Evaluations/Coding Assistant/local"
> UV_CACHE_DIR=uv-cache UV_PYTHON_INSTALL_DIR=uv-python JUPYTER_DATA_DIR=home/jupyter uv run --no-project --python 3.12 --with jupyterlab jupyter lab "05 Local mode walkthrough.ipynb"
> ```
>
> The variables keep Jupyter's own packages and settings in this folder, with the rest of the toolchain, so section 9 removes them too. `JUPYTER_DATA_DIR` also makes sure that this notebook runs on the Python of this command, not on a `python3` kernel that you registered for another project.

### Procedure

1. What changed, and why (3 minutes of reading).
2. Check the prerequisites.
3. Set up the toolchain and the sealed clones.
4. Look at a task and at the grader.
5. See what the sandbox lets the agent reach.
6. Run one PR-review trial and one coding trial.
7. Run the controls: the empty agent scores 0, the oracle scores 1.
8. The set of 12 trials, and how to read the results.
9. Clean up.

## 1. What Changed, and Why

Notebook 01 explains the approach: roll the repository back to the parent commit, seal the environment, and compare the agent's output to the oracle. The Docker path does each of these with a container per trial and Harbor to run them. That needs Docker, Harbor and about 15 GB of images.

This mode does the same jobs with four small pieces on the host:

| Job | Docker and Harbor path | Local mode |
|---|---|---|
| Start state and git seal | the task's Dockerfile clones the base commit, removes the remote, runs `gc` | the same git commands, run once per task on your Mac (`p1 prepare`). Each trial gets a copy of that clone |
| Dependencies | the task image (Debian, Python 3.12) | a `uv` virtual environment per trial, from the task's own `requirements.lock` |
| Network seal | a proxy allowlist | `srt`: the agent can reach the two Bedrock hosts on port 443 and nothing else |
| Hidden tests and oracle | uploaded to the verifier after the agent stops | stay in the module folder, which the agent's sandbox cannot read |
| Verifier | `tests/test.sh` in the container | the same `tests/test.sh`, from a temporary copy with host paths, in a second sandbox that allows only Bedrock |
| Run and record | Harbor, `jobs/<job>/<trial>/` | `runner/p1.py` (Python standard library only), the same `jobs/<job>/<trial>/` layout |

What stays the same: the 4 tasks, the instructions, the hidden tests, the validation agent, the rubric and taste judges, the model IDs in `model_config.py`, and the Codex version and flags.

What the agent sees differently: host paths instead of `/work/repo`, and one sentence of the instruction ("You are inside of a Docker container." becomes "You are working on a laptop, in a sandbox with its own copy of the repository."). Section 4 shows the difference.

```
 Jupyter (started from a plain terminal)
   |
   |  setup.sh        Node 22, srt, Codex, ripgrep  ->  this folder only
   |  p1 prepare      git clone + seal, once per task  ->  ~/.coding-assistant-local/work/base/<task>/
   v
 p1 trial  --  aws sts: agent token (1 h, 3 model IDs), verifier token (2 h)
   |
   |  copy of the sealed clone + uv venv  ->  ~/.coding-assistant-local/work/trials/<trial>/
   v
 +----------------------------- srt sandbox (agent profile) ------------------------------+
 |  Codex CLI   reads and writes only its trial folder                                    |
 |              reaches only bedrock-mantle and bedrock-runtime (us-east-1, port 443)     |
 +----------------------------------------------------------------------------------------+
   |  the agent stops
   v
 +----------------------------- srt sandbox (verifier profile) ---------------------------+
 |  tests/test.sh: hidden tests, base suite, validation agent, rubric and taste judges    |
 |              reaches only bedrock-runtime                                              |
 +----------------------------------------------------------------------------------------+
   |
   v
 jobs/<job>/<trial>/   result.json, verifier/reward.json, agent/codex.txt, agent/sessions/
```

The work folder `~/.coding-assistant-local/work` is outside the module folder on purpose: the module path has spaces ("Workload Specific Evaluations"), and the verifier scripts use paths without quotes.

**What was measured** (one Mac, macOS on Apple silicon, November 2026, n=3 per task; details in [README.md](README.md)): the grader gives the same deterministic reward keys as the Docker path in 11 of 11 coding cases and identical PR-review rewards in 6 of 6. 12 Codex trials gave the same results as the Docker path, with 0 invalid trials. The sandbox held against scripted probes and two LLM red-team runs. This is a preview: it is tested on macOS on Apple silicon only.

## 2. Check the Prerequisites

| Need | Why |
|---|---|
| macOS on Apple silicon | `setup.sh` downloads arm64 builds, and `srt` uses `sandbox-exec` (Seatbelt) |
| Jupyter started from a plain terminal | a sandbox cannot start inside another sandbox |
| The checkout under your home folder | the sandbox denies reads of your home folder. A checkout somewhere else (for example `/opt`) would be readable by the agent, including the hidden tests |
| `uv`, `git`, `curl`, `tar`, AWS CLI v2 | `brew install uv`; the `git`, `curl` and `tar` of macOS are fine. `env.sh` looks for tools in `/opt/homebrew/bin`, `/usr/local/bin` and `/usr/bin` only |
| An IAM user access key in the default AWS profile | the runner calls `aws sts get-session-token` and `get-federation-token`, which AWS does not allow with role or IAM Identity Center credentials. TESTER-GUIDE.md section 2 has a minimal policy |
| Model access in `us-east-1` | the agent model and the judge model in `model_config.py` |
| About 3 GB of free disk | toolchain, uv cache, sealed clones and trial folders |

You do not need Docker, Harbor, `gh`, a system Python or `sudo`.

This first cell sets the paths and one helper, `local()`. It runs a command in this folder in the same way as a new plain terminal after `source env.sh`: your shell's `AWS_*` variables, the Jupyter environment and its virtual environment are not passed on. The runner's output comes back through a pipe and is printed here. (Do not redirect the runner's output to a file yourself: inside the sandbox, `node` aborts when its output is a file in your home folder.)

In [ ]:
import json
import os
import platform
import re
import shlex
import shutil
import signal
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

from IPython.display import Markdown, display

DRY_RUN = os.environ.get("LOCAL_NB_DRY_RUN") == "1"  # maintainers: print the commands instead of running them

LOCAL = Path.cwd()       # this notebook's folder: Coding Assistant/local
MODULE = LOCAL.parent    # the module: datasets/, coding_eval.py, model_config.py
assert (LOCAL / "runner" / "p1.py").exists(), "Open this notebook from the local/ folder of the Coding Assistant module."
REAL_HOME = os.environ.get("P1_REAL_HOME") or str(Path.home())        # env.sh changes HOME; it keeps the real one here
os.environ.setdefault("P1_WORK", f"{REAL_HOME}/.coding-assistant-local/work")  # the same default as env.sh
P1_WORK = Path(os.environ["P1_WORK"])

sys.dont_write_bytecode = True                       # no __pycache__ in the module folder
sys.path[:0] = [str(MODULE), str(LOCAL / "runner")]
import coding_eval as ce                             # the module's helpers (notebooks 02 to 04)
from coding_eval import read_json, read_toml
from model_config import CODEX_MODEL_ID, VERIFIER_MODEL_ID
import p1 as runner                                  # the local runner, imported to read its settings


def mask(text: str) -> str:
    """Hide your home path and any 12-digit number (an AWS account ID) in what this notebook prints."""
    return re.sub(r"(?<!\d)\d{12}(?!\d)", "<12 digits>", text.replace(REAL_HOME, "~"))


_tool_dirs = [str(Path(p).parent) for p in map(shutil.which, ("uv", "aws", "git")) if p]
SHELL_ENV = {"PATH": ":".join(dict.fromkeys(_tool_dirs + ["/opt/homebrew/bin", "/usr/local/bin", "/usr/bin", "/bin", "/usr/sbin", "/sbin"])),
             "HOME": REAL_HOME, "LANG": "en_US.UTF-8", "TERM": "dumb",
             "USER": os.environ.get("USER", ""), "TMPDIR": os.environ.get("TMPDIR", "/tmp")}


def local(cmd: str, check: bool = True, capture: bool = False) -> str | None:
    """Run `cmd` in this folder after `source env.sh`, as in a new plain terminal. Print the output as it arrives.
    Return the output if `capture` is True."""
    print(f"$ {cmd}", flush=True)
    if DRY_RUN:
        return "" if capture else None
    t0 = time.time()
    proc = subprocess.Popen(["/bin/bash", "-c", f"source ./env.sh && {cmd}"], cwd=LOCAL, env=SHELL_ENV, text=True,
                            stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, start_new_session=True)
    lines = []
    try:
        for line in proc.stdout:
            lines.append(line)
            print(mask(line), end="", flush=True)
        rc = proc.wait()
    except KeyboardInterrupt:  # "Interrupt the kernel" stops the runner like Ctrl-C, so it releases its lock
        os.killpg(proc.pid, signal.SIGINT)
        proc.wait()
        raise
    print(f"[exit {rc} after {time.time() - t0:.0f} s]")
    if check and rc != 0:
        raise RuntimeError(f"The command failed (exit {rc}). Read the lines above. TESTER-GUIDE.md section 9 lists the usual causes.")
    return "".join(lines) if capture else None


P1 = "uv run --no-project --python 3.12 runner/p1.py"   # the guide's `p1` shorthand
FED = "--federated scripts/agent-policy.json"            # the agent gets a 1-hour token limited to three model IDs
TASK_NAMES = ["pr371-fix", "pr338-fix", "pr4121-approve", "pr4615-reject"]
print("Module:", MODULE.name, "| work folder:", mask(str(P1_WORK)), "| dry run" if DRY_RUN else "")

### Do the setup checks

Each check prints `OK` or `ERR` and what to do. Run this cell again after you repair an item.

In [ ]:
def run(cmd: list[str], env: dict | None = None, timeout: int = 60) -> tuple[bool, str]:
    """Run a command. Return (ok, output)."""
    try:
        out = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, stdin=subprocess.DEVNULL, env=env or SHELL_ENV)
        return out.returncode == 0, (out.stdout + out.stderr).strip()
    except (FileNotFoundError, subprocess.TimeoutExpired) as error:
        return False, str(error)


ENV_SH_DIRS = ("/opt/homebrew/bin", "/usr/local/bin", "/usr/bin")   # env.sh sets PATH to these (and the toolchain)


def check_platform():
    ok = platform.system() == "Darwin" and platform.machine() == "arm64"
    return ok, f"{platform.system()} {platform.machine()} {platform.mac_ver()[0]}" + ("" if ok else ": this mode needs macOS on Apple silicon")


def check_python():
    ok = sys.version_info >= (3, 11)
    return ok, f"kernel Python {sys.version.split()[0]}" + ("" if ok else ": start Jupyter with --python 3.12 (first cell)")


def check_under_home():
    try:
        LOCAL.resolve().relative_to(Path(REAL_HOME).resolve())
        return True, "the checkout is under your home folder"
    except ValueError:
        return False, ("Move the checkout under your home folder. The sandbox hides your home folder from the agent; "
                       "outside it, the agent could read the hidden tests.")


def check_work_dir():
    ok = " " not in str(P1_WORK)
    return ok, mask(str(P1_WORK)) if ok else "Your home path has a space. Set P1_WORK to a path without spaces before you start Jupyter."


def check_tool(name: str, args: list[str], hint: str):
    path = shutil.which(name, path=SHELL_ENV["PATH"])
    if not path:
        return False, f"{name} is not on PATH. {hint}"
    if str(Path(path).parent) not in ENV_SH_DIRS:
        return False, f"{name} is in {mask(str(Path(path).parent))}, which env.sh does not search. {hint}"
    ok, out = run([path, *args])
    return ok, (out.splitlines() or [""])[0]


def check_aws_v2():
    ok, detail = check_tool("aws", ["--version"], "Install AWS CLI v2: https://docs.aws.amazon.com/cli/latest/userguide/getting-started-install.html")
    return ok and detail.startswith("aws-cli/2"), detail if detail.startswith("aws-cli/2") or not ok else f"{detail}: the runner needs AWS CLI v2"


def check_sandbox():
    ok, out = run(["/usr/bin/sandbox-exec", "-p", "(version 1)(allow default)", "/usr/bin/true"])
    if ok:
        return True, "a sandbox can start from this kernel"
    return False, (f"a sandbox cannot start here ({out[-100:]}). Jupyter runs inside another sandbox. Stop it, "
                   "open Terminal.app or iTerm, start Jupyter there with the command in the first cell, and run this cell again.")


checks = {
    "macOS, Apple silicon": check_platform(),
    "Python 3.11+":         check_python(),
    "plain terminal":       check_sandbox(),
    "checkout under ~":     check_under_home(),
    "work folder":          check_work_dir(),
    "uv":                   check_tool("uv", ["--version"], "Install it with `brew install uv`."),
    "git":                  check_tool("git", ["--version"], "Install the Xcode command line tools: `xcode-select --install`."),
    "curl":                 check_tool("curl", ["--version"], "curl ships with macOS."),
    "tar":                  check_tool("tar", ["--version"], "tar ships with macOS."),
    "AWS CLI v2":           check_aws_v2(),
}
for name, (ok, detail) in checks.items():
    print(f"{'OK ' if ok else 'ERR'} {name:<22} {detail}")
failed = [name for name, (ok, _) in checks.items() if not ok]
if failed and not DRY_RUN:
    raise RuntimeError(f"Repair these items, then run this cell again: {', '.join(failed)}")
print("\nAll checks passed." if not failed else "\nDry run: checks not enforced.")

### Check your AWS identity and Bedrock access

The runner reads only the **default** AWS profile (it ignores `AWS_PROFILE` and the `AWS_*` variables), so this check does the same. It prints the type of the identity, not its name or account. Then it makes one short call to the judge model. Codex's model is checked from inside the sandbox in section 5 (stage L6a).

In [ ]:
if DRY_RUN:
    print("Dry run: no AWS calls.")
else:
    ok, out = run(["aws", "sts", "get-caller-identity", "--output", "json"])
    if not ok:
        raise RuntimeError("The default AWS profile has no working credentials: " + mask(out[-300:]))
    kind = json.loads(out)["Arn"].split(":")[5].split("/")[0]  # user, assumed-role, federated-user or root
    print("Default profile identity type:", kind)
    if kind != "user":
        raise RuntimeError("The runner needs an IAM user access key in the default profile. Role and IAM Identity Center "
                           "credentials cannot call get-federation-token (TESTER-GUIDE.md section 2).")

    message = json.dumps([{"role": "user", "content": [{"text": "Reply with the single word: yes"}]}])
    ok, out = run(["aws", "bedrock-runtime", "converse", "--region", runner.REGION, "--model-id", VERIFIER_MODEL_ID,
                   "--messages", message, "--inference-config", '{"maxTokens": 256}',
                   "--query", "output.message.content[0].text", "--output", "text"])
    if not ok:
        raise RuntimeError(f"Cannot call {VERIFIER_MODEL_ID} in {runner.REGION}: {mask(out[-300:])}\n"
                           "Request access to the model in the Amazon Bedrock console.")
    print(f"OK  {VERIFIER_MODEL_ID} replied: {out.strip()[:40]}")

## 3. Set Up the Toolchain

`setup.sh` downloads Node 22, `srt` (the sandbox), Codex CLI and ripgrep, with pinned versions, and writes a small `timeout` shim (macOS has no GNU `timeout`, and the verifier calls it). Everything goes into this folder:

| Folder | Content | Size |
|---|---|---|
| `node/`, `npm/`, `npm-cache/` | Node 22, `srt`, Codex CLI | about 0.7 GB |
| `bin/` | `rg`, `timeout` | 4 MB |
| `uv-cache/`, `uv-python/` | Python 3.12 and the task packages, filled by the first environment build | about 1.1 GB |
| `home/` | the `HOME` of the toolchain (`env.sh` sets it) | under 1 MB |

Nothing is installed system-wide, nothing needs `sudo`, and git ignores all of it. Measured: 36 s the first time, 1 s when you run it again.

In [ ]:
local("./setup.sh")

### Prepare the sealed clones

`p1 prepare` runs the git steps of each task's Dockerfile on your Mac: clone the repository at the base commit, remove the remote and the remote refs, expire the reflog and prune everything that is not reachable. For a PR-review task it then applies the change under review as uncommitted edits. Each trial later gets a copy-on-write copy of this clone, so the agent always starts from the same state.

Measured: about 30 s for the four tasks the first time. A task that is already prepared is skipped (0 s).

In [ ]:
local(f"{P1} prepare {' '.join(TASK_NAMES)}")

## 4. Look at a Task and the Grader

The tasks are the module's own, in `datasets/`. Notebooks 02 and 03 explain them in detail. Here you look at the parts that matter for this mode. The runner's `Task` class reads a task directory: where the repository is, its base commit, and if it is a coding task or a PR-review task.

In [ ]:
TASKS = {name: runner.Task(name) for name in TASK_NAMES}
for name, t in TASKS.items():
    print(f"{name:<15} {t.kind:<6} {t.url}  base {t.sha[:9]}  change under review: {t.has_change}")

### The coding datapoint

A coding datapoint is a real bug: an issue report, the commit before the fix, and the fix that the team merged. `[metadata]` in `task.toml` records where it came from, the hidden tests, and the cause and the fix. The agent does not see `task.toml`.

In [ ]:
t = TASKS["pr371-fix"]
meta = read_toml(t.dir / "task.toml")["metadata"]
origin, verifier, story = meta["origin"], meta["verifier"], meta["narrative"]
print(t.full)
print(f"  origin: {origin['repo']} PR {origin['pr_numbers']}, issue {origin['issue_numbers']}, base {origin['base_commit'][:9]}")
print(f"  tests:  {verifier['fail_to_pass']} fail-to-pass, {verifier['pass_to_pass']} pass-to-pass, in {verifier['hidden_tests']}")
print(f"  cause:  {' '.join(story['description'].split())}")
print(f"  fix:    {' '.join(story['solution'].split())}")

### The same git steps as the Dockerfile

These are the git lines of the task's Dockerfile. `p1 prepare` runs the same commands (see `prepare()` in `runner/p1.py`).

In [ ]:
dockerfile = (t.dir / "environment" / "Dockerfile").read_text()
for line in dockerfile.splitlines():
    if "git " in line and not line.lstrip().startswith("#"):
        print(line.strip()[:150])

### The network allowlist

`task.toml` sets the network of each phase for Harbor. The runner gives `srt` the same hosts, with the port.

In [ ]:
cfg = read_toml(t.dir / "task.toml")
print("task.toml, agent:    ", cfg["environment"].get("allowed_hosts"))
print("task.toml, verifier: ", cfg["verifier"].get("allowed_hosts"))
print("runner, agent:       ", runner.AGENT_HOSTS)
print("runner, verifier:    ", runner.VERIFIER_HOSTS)

### What the agent reads

The instruction is the task's `instruction.md`. The runner changes only the container paths and one sentence. This diff is the whole difference from the Docker path:

In [ ]:
for name in ("pr371-fix", "pr4121-approve"):
    _, diff = runner.render_instruction(TASKS[name], Path("<trial folder>"))
    print(f"== {name}")
    print("\n".join(line[:160] for line in diff), "\n")

### The grader

The verifier files are in `tests/` of each task. The agent never sees them: they stay in the module folder, which its sandbox cannot read. After the agent stops, the runner copies `tests/` to a temporary folder, replaces the container paths (`/work/repo`, `/tests`, `/logs`, `/tmp`) with the trial's host paths, and runs `tests/test.sh`. The files in `datasets/` do not change.

In [ ]:
ce.show_tree(t.dir / "tests")

For a coding task, `test.sh` runs the reward stack of notebook 03:

| Stage | What it checks | LLM? | Gate for a tasteful solve |
|---|---|---|---|
| Hidden tests | fail-to-pass and pass-to-pass tests, and the base unit test suite | No | all pass |
| Validation agent | tests that an agent writes from the validation spec, for the shape of the agent's change | Yes | all stories pass |
| Task rubric judge | task-specific criteria that tests cannot check well | Yes | score > 0.5 |
| Taste judge | five yes/no checks against the merged fix, five for fit with the codebase | Yes | at least 3 of 5 in each group |
| Patch bloat | size of the change compared with the merged fix | No | < 2x |

`correct` means that the change works (tests and validation stories). `reward` (a tasteful solve) also needs the rubric, taste and bloat gates.

For a PR-review task (notebook 02), an LLM judge matches the agent's findings to the oracle findings. `coverage_required` is the share of the high-severity oracle findings that the agent found, and `coverage_all` the share of all of them. A review passes if the verdict is right: on the approve task, no `high` finding; on the reject task, a `high` finding and `coverage_required` of 1.0.

## 5. See What the Sandbox Lets the Agent Reach

The seal is only as good as the sandbox. This section looks at the sandbox profile, then runs the cheap stages of the test ladder and one extra probe.

### The agent's sandbox profile

The runner writes this profile for each trial. Reads of your home folder, `/Users`, `/Volumes` and `/tmp` are denied, except for the trial folder and the toolchain. Writes are allowed only in the trial folder. The network goes through `srt`'s proxy, which allows only the listed hosts.

In [ ]:
W = runner.WORK / "trials" / f"{t.full}__r1"
profile = runner.srt_settings(W, runner.AGENT_HOSTS, [W], [W], REAL_HOME)
print(mask(json.dumps(profile, indent=1)))

### Run the cheap stages of the test ladder

`run_all.sh` runs the test ladder of TESTER-GUIDE.md section 5. Three stages are cheap:

- **L0** starts the sandbox: `srt -- true`, then `echo sandbox-ok` inside it. If it prints `sandbox_apply: Operation not permitted`, Jupyter runs inside another sandbox.
- **L2** runs `scripts/l2_inner.sh` inside the agent's sandbox: both Bedrock hosts must answer, and GitHub, PyPI, npm, plain HTTP, IP addresses, IPv6, UDP, DNS and raw sockets must be blocked.
- **L6a** starts `node`, then asks Codex to reply "OK" inside the agent's sandbox. This checks that Codex can reach its model on Bedrock. If it cannot, the stage tries other settings and says which one works.

About 2 minutes, a few cents. `run_all.sh` writes each stage's log to `logs/run_all/<stage>.log`.

In [ ]:
local('STAGES="L0 L2 L6a" ./run_all.sh')

In [ ]:
def table(header: list[str], rows: list[list]) -> None:
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(c).replace("|", "/") for c in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


l0 = LOCAL / "logs" / "run_all" / "L0.log"
print("L0:", l0.read_text().strip() if l0.exists() else "no log yet (run the cell above)")
gates = LOCAL / "logs" / "L2-gates.txt"
rows = []
for line in (gates.read_text().splitlines() if gates.exists() else []):
    if line.startswith("GATE|"):
        _, name, expected, observed, _detail = (line.split("|") + [""] * 5)[:5]
        if expected.startswith("reachable"):
            good = observed.startswith("reachable")
        elif expected == ">0":
            good = observed.strip().isdigit() and int(observed) > 0
        else:
            good = observed.startswith("blocked")
        rows.append(["OK" if good else "CHECK", name, expected, mask(observed)])
table(["", "L2 network gate", "expected", "observed"], rows) if rows else print("No L2 gates yet (run the cell above).")

### What the agent can and cannot read

This probe runs inside the agent's sandbox for the coding task, through the runner's `sandbox-run` command (the same command that stage L2 uses). It tries to read a hidden test and the merged fix in the module folder, to list your home folder and `~/.ssh`, and to write outside its trial folder. It reads nothing from these places: it prints only `readable`, `listed`, `written` or `denied`. Then it checks the git seal from the agent's point of view.

The probe script is written to `logs/`, which git ignores.

In [ ]:
hidden = next((t.dir / "tests" / "verify").glob("*_verify.py"))
oracle_patch = t.dir / "solution" / "oracle.patch"
probe = f"""#!/bin/bash
# Runs inside the agent's sandbox. One line per check: CHECK|what|expected|observed
c() {{ echo "CHECK|$1|$2|$3"; }}
r() {{ cat "$1" > /dev/null 2>&1 && echo readable || echo denied; }}
l() {{ ls "$1" > /dev/null 2>&1 && echo listed || echo denied; }}
c "read a hidden test in the module folder" denied "$(r {shlex.quote(str(hidden))})"
c "read the merged fix in the module folder" denied "$(r {shlex.quote(str(oracle_patch))})"
c "list your home folder" denied "$(l "$P1_REAL_HOME")"
c "list ~/.ssh" denied "$(l "$P1_REAL_HOME/.ssh")"
c "write a file in your home folder" denied "$(touch "$P1_REAL_HOME/.p1-nb-probe" 2>/dev/null && rm -f "$P1_REAL_HOME/.p1-nb-probe" && echo written || echo denied)"
c "write in its own trial folder" written "$(touch ../tmp/probe 2>/dev/null && echo written || echo denied)"
c "commits in the clone (git log --all)" 1 "$(git log --all --oneline 2>/dev/null | wc -l | tr -d ' ')"
c "git remotes in the clone" 0 "$(git remote 2>/dev/null | wc -l | tr -d ' ')"
c "hidden tests or oracle files in the trial folder" 0 "$(find .. -name '*_verify.py' -o -name oracle.patch -o -name reference.patch 2>/dev/null | wc -l | tr -d ' ')"
c "HTTPS to github.com" blocked "$(curl -fsS -o /dev/null --max-time 10 https://github.com/ 2>/dev/null && echo REACHED || echo blocked)"
"""
(LOCAL / "logs").mkdir(exist_ok=True)
(LOCAL / "logs" / "nb-probe.sh").write_text(probe)

out = local(f"{P1} sandbox-run pr371-fix --tag nbprobe --script logs/nb-probe.sh", check=False, capture=True)
rows = [line.split("|")[1:4] for line in out.splitlines() if line.startswith("CHECK|")]
table(["what the agent tries", "expected", "observed", ""], [[w, e, o, "OK" if o == e else "CHECK"] for w, e, o in rows]) if rows else None
for leftover in runner.WORK.glob("trials/*__nbprobe*"):   # the probe's trial folder is not needed after this
    shutil.rmtree(leftover, ignore_errors=True) if leftover.is_dir() else leftover.unlink()

A `CHECK` in the last column means the observed value is not the expected one. Do not run an agent until you know why: send the table with the feedback template at the end of TESTER-GUIDE.md.

### The sealed clones, from the host

The same seal, checked on the four sealed clones from outside the sandbox: one commit, no remote, no reflog, and no hidden test file.

In [ ]:
rows = []
for name, task in TASKS.items():
    repo = task.base / "repo"
    if not repo.is_dir():
        rows.append([name, "not prepared (run section 3)", "", "", "", ""])
        continue
    git = lambda *args: subprocess.run(["git", *args], cwd=repo, capture_output=True, text=True, env=SHELL_ENV).stdout
    commits, remotes = len(git("log", "--all", "--oneline").splitlines()), len(git("remote").split())
    reflog, hidden_files = len(git("reflog", "--all").splitlines()), len(list(repo.rglob("*_verify.py")))
    ok = commits == 1 and remotes == 0 and reflog == 0 and hidden_files == 0
    rows.append([name, "OK" if ok else "CHECK", commits, remotes, reflog, hidden_files])
table(["task", "seal", "commits", "remotes", "reflog entries", "hidden test files"], rows)

## 6. Run One PR-Review Trial and One Coding Trial

`p1 trial` does one trial, like one container in Harbor:

1. Mint two short-lived credentials with `aws sts`: a 1-hour federated token for the agent, limited by `scripts/agent-policy.json`, and a 2-hour session token for the verifier. They are passed only in the environment of the processes that need them, and never written to disk.
2. Start the environment: a copy of the sealed clone and a fresh `uv` virtual environment.
3. Run Codex inside the agent's sandbox, with the same version and flags as Harbor's Codex adapter.
4. After Codex stops, run the verifier inside its own sandbox.
5. Record the trial in `jobs/<job>/<trial>/`, then delete the trial folder.

The runner takes the lock `.bedrock.lock`, so only one trial runs at a time. A trial with the same job and trial name replaces the earlier record, so you can run these cells again.

### PR review: the approve task

Measured: about 52 s in total, about $0.13 for the agent.

In [ ]:
local(f"{P1} trial pr4121-approve --agent codex --job nb {FED}")

This cell reads the record: the timings of each phase, what the agent used, and the reward.

In [ ]:
def seconds_between(start: str, end: str) -> float:
    parse = lambda s: datetime.fromisoformat(s.replace("Z", "+00:00"))
    return (parse(end) - parse(start)).total_seconds()


def trial_dir(job: str, name: str, tag: str = "r1") -> Path:
    return LOCAL / "jobs" / job / f"{TASKS[name].full}__{tag}"


def show_trial(job: str, name: str, tag: str = "r1") -> dict | None:
    """Print the timings, the agent's usage and the reward of one recorded trial. Return the reward."""
    d = trial_dir(job, name, tag)
    if not (d / "result.json").exists():
        print(f"No record at jobs/{job}/{d.name}. Run the trial cell first.")
        return None
    res = read_json(d / "result.json")
    env, agent, ver = res.get("environment_setup", {}), res.get("agent_result") or {}, res.get("verifier") or {}
    print(f"== jobs/{job}/{d.name}")
    print(f"  total {seconds_between(res['started_at'], res['finished_at']):.0f} s: environment {env.get('clone_s', 0) + env.get('venv_s', 0):.0f} s, "
          f"agent {agent.get('seconds', 0):.0f} s, verifier {ver.get('seconds', 0):.0f} s")
    if agent:
        print(f"  agent: exit {agent.get('exit')}, {agent.get('steps')} steps, {agent.get('tool_calls')} tool calls, "
              f"{agent.get('input_tokens')} input tokens ({agent.get('cached_input_tokens')} cached), {agent.get('output_tokens')} output tokens")
    if ver.get("stages"):
        print("  verifier stages (s):", ver["stages"])
    reward_file = d / "verifier" / "reward.json"
    if not reward_file.exists():
        details = d / "verifier" / "reward_details.json"
        why = read_json(details).get("invalid") if details.exists() else "no verifier output"
        print(f"  INVALID trial (no reward.json): {why}. An invalid trial is not a score of 0.")
        return None
    reward = read_json(reward_file)
    print("  reward:", {k: v for k, v in reward.items() if isinstance(v, (int, float))})
    return reward


reward = show_trial("nb", "pr4121-approve")
findings_file = trial_dir("nb", "pr4121-approve") / "artifacts" / "logs" / "artifacts" / "findings.json"
if reward is not None:
    findings = read_json(findings_file) if findings_file.exists() else []
    verdict = "block" if any(f.get("severity") == "high" for f in findings) else "approve"
    print(f"\n  verdict: {verdict} (expected: approve), {len(findings)} findings")
    for f in findings:
        print(f"    [{f.get('severity')}] {f.get('path', '').split('/')[-1]}:{f.get('line')} {f.get('statement', '')[:110]}")

### Coding: the root agent bug

Measured: about 5 minutes in total (agent about 85 s, verifier about 3 minutes, of which the hidden and base tests are about 100 s), about $0.21 for the agent and $0.02 to $0.03 for the judges.

In [ ]:
local(f"{P1} trial pr371-fix --agent codex --job nb {FED}")

In [ ]:
SHOW = ["reward", "correct", "fail_to_pass", "pass_to_pass", "existing_suite", "validation_score",
        "rubric_score", "relative_taste", "practice_alignment", "patch_bloat"]
reward = show_trial("nb", "pr371-fix")
details_file = trial_dir("nb", "pr371-fix") / "verifier" / "reward_details.json"
if reward and details_file.exists():
    details = read_json(details_file)
    status = "TASTEFUL SOLVE" if reward.get("reward") == 1.0 else ("CORRECT, NOT TASTEFUL" if reward.get("correct") == 1.0 else "FAIL")
    print(f"\n  {status}")
    for gate, ok in details.get("checks", {}).items():
        print(f"    {'pass' if ok else 'FAIL'}  {gate}")
    print("  ", {k: reward.get(k) for k in SHOW})

### Open the agent's patch and its log

Each record has the same files as a Harbor trial, except `trajectory.json`:

| File | Content |
|---|---|
| `verifier/agent.patch` | the agent's change as the verifier saw it |
| `verifier/reward_details.json` | every gate, failed test, validation story, rubric criterion and taste check, with the judges' reasons |
| `verifier/test.sh.out` | the verifier's output. The line `could not write config file .git/config` is expected: the sandbox denies it, and the runner sets the git identity another way |
| `verifier/render.diff` | the path substitutions in the verifier files |
| `agent/codex.txt` | Codex's JSON event stream. `Reconnecting... (Connection failed ...)` lines mean that it could not reach Bedrock |
| `agent/sessions/` | Codex's own session log, with every step |

This cell prints the start of the agent's patch next to the merged fix, and the first commands that Codex ran.

In [ ]:
d = trial_dir("nb", "pr371-fix")
if (d / "verifier" / "agent.patch").exists():
    print(f"Record folder: {mask(str(d.relative_to(LOCAL)))}\n")
    print((d / "verifier" / "agent.patch").read_text()[:2000])
    print("\n-- merged fix --")
    print((TASKS["pr371-fix"].dir / "solution" / "oracle.patch").read_text()[:2000])
    print("\n-- the first commands of the agent --")
    commands = []
    for line in (d / "agent" / "codex.txt").read_text(errors="replace").splitlines():
        try:
            event = json.loads(line)
        except ValueError:
            continue
        item = event.get("item") or {}
        if event.get("type") == "item.completed" and item.get("type") == "command_execution":
            commands.append(str(item.get("command", "")).replace("\n", " ")[:150])
    print("\n".join(f"  $ {mask(c)}" for c in commands[:12]), f"\n  ({len(commands)} commands in total)")
else:
    print("No record yet. Run the coding trial first.")

**Questions to answer:**

1. The agent's patch is larger than the merged fix. Which part of it is a test that the agent wrote? Does `agent_tests_reproduce_bug` say that the test fails on the base commit?
2. Compare the agent seconds with the verifier seconds. Which phase would you make faster first, and how?

## 7. Run the Controls

Before you trust a score, check the grader with two agents whose answer you know (notebook 01, section 5):

- **`nop`**, the empty agent, changes nothing. It must score 0.
- **`oracle`** submits the reference answer. It must score 1. If it does not, the grader has a defect.

No model runs for the agent, so a control costs only the judge calls. On the reject PR-review task each control takes about 15 s (measured).

In [ ]:
for agent in ("oracle", "nop"):
    local(f"{P1} trial pr4615-reject --agent {agent} --job nb-ctl --tag {agent}")

expected = {"oracle": 1.0, "nop": 0.0}
for agent, value in expected.items():
    reward = show_trial("nb-ctl", "pr4615-reject", agent)
    if reward is not None:
        assert reward.get("coverage_all") == value, f"The {agent} control must have coverage_all {value}. The grader is broken."
        print(f"  OK: {agent} has coverage_all {value}\n")

On the **approve** task, the empty agent gets coverage 1.0, because there is no defect to find and an empty review approves. That is why the controls use the reject task.

The coding controls take about 4 minutes each, because the verifier runs the test suites and the judges. Set `RUN_CODING_CONTROLS = True` to run them on the coding task: the oracle must have `reward` 1, and the empty patch `reward` 0.

In [ ]:
RUN_CODING_CONTROLS = False  # optional: about 8 minutes, a few cents of judge calls

if RUN_CODING_CONTROLS:
    for agent in ("oracle", "nop"):
        local(f"{P1} trial pr371-fix --agent {agent} --job nb-ctl --tag {agent}")
    for agent, value in {"oracle": 1.0, "nop": 0.0}.items():
        reward = show_trial("nb-ctl", "pr371-fix", agent)
        if reward is not None:
            assert reward.get("reward") == value, f"The {agent} control must have reward {value}. See verifier/reward_details.json."
            print(f"  OK: {agent} has reward {value}\n")
else:
    print("Skipped. Set RUN_CODING_CONTROLS = True to run the coding controls.")

## 8. The Set of 12 Trials, and How to Read It

One trial per task shows that the pipeline works. It does not rank agents. The measured set is 4 tasks x 3 Codex trials, one at a time: **37.7 minutes, about $2.60** (agent $2.46 and judges about $0.15, estimate). The command is the one in TESTER-GUIDE.md:

```bash
p1 job --tasks pr371-fix,pr338-fix,pr4121-approve,pr4615-reject --n 3 --agent codex --job nb-full --federated scripts/agent-policy.json
```

`./run_all.sh` with `STAGES=L7` runs the same set and then the oracle and empty controls on all four tasks (about 54 minutes). You can run either in a plain terminal instead of here, after `source env.sh`.

`RUN_SLICE` runs a slice of 3 trials, one per task type, in about 7 minutes (about $0.55). `RUN_FULL_SET` runs the 12. Both are off.

In [ ]:
RUN_SLICE = False     # 3 trials: approve, reject, one coding task. About 7 minutes, about $0.55
RUN_FULL_SET = False  # 12 trials: 4 tasks x 3. About 38 minutes, about $2.60

if RUN_SLICE:
    local(f"{P1} job --tasks pr4121-approve,pr4615-reject,pr371-fix --n 1 --agent codex --job nb-slice {FED}")
if RUN_FULL_SET:
    local(f"{P1} job --tasks {','.join(TASK_NAMES)} --n 3 --agent codex --job nb-full {FED}")
if not (RUN_SLICE or RUN_FULL_SET):
    print("Skipped. Set RUN_SLICE or RUN_FULL_SET to True to run more trials.")

### Read the results

This cell reads every Codex job in `jobs/` that this notebook or `run_all.sh` made, one row per trial. A trial is **valid** only if the agent exited 0, used input tokens and the verifier wrote `reward.json`. The runner stops a job at the first trial that is not valid.

In [ ]:
def trial_rows(job_dir: Path) -> list[list]:
    rows = []
    for result_file in sorted(job_dir.glob("*/result.json")):
        d, res = result_file.parent, read_json(result_file)
        agent = res.get("agent_result") or {}
        name = res["task_name"].split("__")[-1]
        reward = read_json(d / "verifier" / "reward.json") if (d / "verifier" / "reward.json").exists() else None
        valid = agent.get("exit") == 0 and (agent.get("input_tokens") or 0) > 0 and reward is not None
        if not valid:
            score = "INVALID"
        elif "coverage_all" in reward:
            findings_file = d / "artifacts" / "logs" / "artifacts" / "findings.json"
            high = any(f.get("severity") == "high" for f in (read_json(findings_file) if findings_file.exists() else []))
            passed = (not high) if name.endswith("approve") else (high and reward.get("coverage_required") == 1.0)
            score = f"{'pass' if passed else 'fail'} (coverage_required {reward.get('coverage_required')})"
        else:
            score = f"reward {reward.get('reward')}, correct {reward.get('correct')}, taste {reward.get('relative_taste')}"
        rows.append([job_dir.name, d.name.split("__")[-2] if "__" in d.name else name, d.name.split("__")[-1],
                     score, agent.get("seconds"), f"{seconds_between(res['started_at'], res['finished_at']):.0f}"])
    return rows


rows = []
for job in ("nb", "nb-slice", "nb-full", "L6", "L7"):
    if (LOCAL / "jobs" / job).is_dir():
        rows += trial_rows(LOCAL / "jobs" / job)
table(["job", "task", "trial", "result", "agent s", "total s"], rows) if rows else print("No Codex trials recorded yet.")

**How to read these numbers.** The measured set gave reward 1 in 6 of 6 coding trials, pass on the approve task in 3 of 3, and fail on the reject task in 3 of 3, the same as the Docker path. Keep these limits in mind:

- **The judges are noisy.** For the same patch, the taste judge's score moves with a standard deviation of about 0.19 per call (0.4 to 1.0 over 27 repeats). About 5 to 7% of trials on a pr338-style patch get reward 0 from the taste gate alone, in every mode. With 3 trials per task, the chance of at least one such 0 is about 20%. One 0 is not evidence that a mode or an agent is worse.
- **The reject task is hard.** Codex found its defect in 12% of 185 trials in another mode. With that rate, 0 of 3 has a probability of about 0.67. About 85 trials per design are needed to tell 12% from 30%.
- **n=3 per task shows only large differences.** A coding reward that is 1 almost every time cannot show that one setup is worse.

The comparison of the three modes is in [docs/lite-modes-comparison.md](../docs/lite-modes-comparison.md).

## 9. Clean Up

**Nothing exists in AWS.** The runner made only `sts` calls (temporary credentials, which expire on their own after 1 and 2 hours) and Bedrock invocations. There is no resource to delete. If model invocation logging is on in your account, these calls are logged there like any other Bedrock call.

On your Mac, the trial data is in two places, and the toolchain is in this folder:

| Path | Content |
|---|---|
| `~/.coding-assistant-local/work` | sealed clones, trial folders and verifier copies (can grow to several GB) |
| `jobs/`, `logs/`, `.bedrock.lock` in this folder | the records and logs |
| `node/`, `npm/`, `npm-cache/`, `uv-cache/`, `uv-python/`, `home/`, `bin/` in this folder | the toolchain and caches, and Jupyter's own packages if you started it with the command in the first cell |

This cell shows the sizes. Set `DELETE_TRIAL_DATA = True` to delete the first two rows. The toolchain is still in use by this kernel, so delete it from a terminal after you stop Jupyter, with the command that the cell prints. Git ignores all these paths, so `git status` stays clean.

In [ ]:
DELETE_TRIAL_DATA = False  # True deletes the work folder, jobs/, logs/ and the lock

def size(path: Path) -> str:
    if not path.exists():
        return "not present"
    ok, out = run(["du", "-sh", str(path)])
    return out.split()[0] if ok and out else "?"

TRIAL_DATA = [P1_WORK, LOCAL / "jobs", LOCAL / "logs", LOCAL / ".bedrock.lock"]
TOOLCHAIN = ["node", "npm", "npm-cache", "uv-cache", "uv-python", "home", "bin"]
for path in TRIAL_DATA + [LOCAL / name for name in TOOLCHAIN]:
    print(f"{size(path):>12}  {mask(str(path))}")

if DELETE_TRIAL_DATA:
    for path in TRIAL_DATA:
        shutil.rmtree(path, ignore_errors=True)
    for empty in (P1_WORK.parent, Path("/tmp/claude")):   # empty parent of the work folder; srt's shared temp folder
        try:
            empty.rmdir()
        except OSError:
            pass
    print("\nDeleted the trial data.")

print("\nAfter you stop Jupyter, delete the toolchain from this folder in a terminal:")
print(f"  rm -rf {' '.join(TOOLCHAIN)}")

## Summary

- The local mode runs the same tasks, agent and grader as the Docker path, with a sealed git clone, a `uv` environment and an `srt` sandbox per trial, and one small runner.
- Start Jupyter (or the runner) from a plain terminal, and keep the checkout under your home folder.
- Check the sandbox before you trust a score: the L2 network gates, the read probe and the seal.
- Check the grader with the controls: the empty agent scores 0, the oracle scores 1.
- One trial per task shows that the pipeline works. To compare agents or modes, run more trials, and expect judge noise.

**Next:** run the whole test ladder with `./run_all.sh` in a plain terminal ([TESTER-GUIDE.md](TESTER-GUIDE.md), section 5), and send your results with the feedback template at the end of the guide. To look at recorded Docker-path runs without any setup, open [04 Replay recorded runs (no Docker)](../04%20Replay%20recorded%20runs%20%28no%20Docker%29.ipynb).